<a href="https://colab.research.google.com/github/2995250328/RL/blob/main/notebooks/bonus-unit1/bonus-unit1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 附加单元 1: 让我们训练 Huggy 狗狗 🐶 去捡木棒

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit2/thumbnail.png" alt="Bonus Unit 1Thumbnail">

在本笔记本中，我们将通过 **教 Huggy 狗狗捡木棒并直接在浏览器中与它玩耍** 来巩固我们在第一单元学到的知识。

⬇️ 这是你在此单元结束时将实现的目标示例 ⬇️ (点击 ▶ 运行查看)

In [ ]:
%%html
<video controls autoplay><source src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.mp4" type="video/mp4"></video>

### 环境 🎮

- Huggy the Dog，由 [Thomas Simonini](https://twitter.com/ThomasSimonini) 基于 [Puppo The Corgi](https://blog.unity.com/technology/puppo-the-corgi-cuteness-overload-with-the-unity-ml-agents-toolkit) 创建。

### 使用的库 📚

- [MLAgents](https://github.com/Unity-Technologies/ml-agents)

我们一直在努力改进我们的教程，所以 **如果你在这个笔记本中发现了一些问题**，请 [在 Github 仓库上提交 issue](https://github.com/huggingface/deep-rl-class/issues)。

## 本笔记本的目标 🏆

在本笔记本结束时，你将：

- 理解 **用于训练 Huggy 的状态空间、动作空间和奖励函数**。
- **训练你自己的 Huggy** 去捡木棒。
- 能够 **直接在浏览器中与你训练好的 Huggy 玩耍**。

## 本笔记本来自深度强化学习课程
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/deep-rl-course-illustration.jpg" alt="Deep RL Course illustration"/>

在这个免费课程中，你将：

- 📖 在 **理论与实践** 中学习深度强化学习。
- 🧑‍💻 学习 **使用著名的深度强化学习库**，如 Stable Baselines3、RL Baselines3 Zoo、CleanRL 和 Sample Factory 2.0。
- 🤖 在 **独特的环境中训练智能体**

更多内容请查看 📚 教学大纲 👉 https://simoninithomas.github.io/deep-rl-course

不要忘记 **<a href="http://eepurl.com/ic5ZUD">注册课程</a>** (我们收集你的电子邮件是为了在每个单元发布时向你发送链接，并告知你有关挑战和更新的信息)。

保持联系的最佳方式是加入我们的 Discord 服务器，与社区和我们交流 👉🏻 https://discord.gg/ydHrjt3WP5

## 前提条件 🏗️

在深入笔记本之前，你需要：

🔲 📚 通过完成第一单元，**建立对强化学习基础的理解** (MC, TD, 奖励假设...)

🔲 📚 通过完成附加单元 1，**阅读 Huggy 的介绍**

## 设置 GPU 💪
- 为了 **加速智能体的训练，我们将使用 GPU**。为此，请前往 `Runtime > Change Runtime type` (运行时 > 更改运行时类型)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step1.jpg" alt="GPU Step 1">

- 选择 `Hardware Accelerator > GPU` (硬件加速器 > GPU)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/gpu-step2.jpg" alt="GPU Step 2">

## 克隆仓库 🔽

- 我们需要克隆包含 **ML-Agents** 的仓库。

In [ ]:
%%capture
# Clone the repository (can take 3min)
!git clone --depth 1 https://github.com/Unity-Technologies/ml-agents

## 设置虚拟环境 🔽
- 为了让 **ML-Agents** 在 Colab 中成功运行，Colab 的 Python 版本必须满足该库的 Python 要求。

- 我们可以在 `setup.py` 文件中的 `python_requires` 参数下查看支持的 Python 版本。这些文件对于设置 **ML-Agents** 库以供使用是必需的，可以在以下位置找到：
  - `/content/ml-agents/ml-agents/setup.py`
  - `/content/ml-agents/ml-agents-envs/setup.py`

- Colab 当前的 Python 版本（可以使用 `!python --version` 检查）可能与库的 `python_requires` 参数不匹配。因此，安装可能会默默失败，并在稍后执行相同命令时导致如下错误：
  - `/bin/bash: line 1: mlagents-learn: command not found`
  - `/bin/bash: line 1: mlagents-push-to-hf: command not found`

- 为了解决这个问题，我们将创建一个 Python 版本与 **ML-Agents** 库兼容的虚拟环境。

`注意:` *为了未来的兼容性，请务必检查安装文件中的 `python_requires` 参数。如果 Colab 的 Python 版本不兼容，请在下方的脚本中将虚拟环境设置为支持的最高 Python 版本。*

In [ ]:
# Colab's Current Python Version (Incompatible with ML-Agents)
!python --version

In [ ]:
# Install virtualenv and create a virtual environment
!pip install virtualenv
!virtualenv myenv

# Download and install Miniconda
!wget https://repo.anaconda.com/miniconda/Miniconda3-latest-Linux-x86_64.sh
!chmod +x Miniconda3-latest-Linux-x86_64.sh
!./Miniconda3-latest-Linux-x86_64.sh -b -f -p /usr/local

# Accept Anaconda Terms of Service to allow package installation
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/main
!conda tos accept --override-channels --channel https://repo.anaconda.com/pkgs/r

# Activate Miniconda and install Python ver 3.10.12
!source /usr/local/bin/activate
!conda install -q -y --prefix /usr/local python=3.10.12 ujson

# Set environment variables for Python and conda paths
os.environ['PYTHONPATH'] = '/usr/local/lib/python3.10/site-packages/'
os.environ['PATH'] = "/usr/local/bin:" + os.environ['PATH']

In [ ]:
# Verify the Python version in the current path
import os
os.environ['PYTHONPATH'] = '/usr/local/lib/python3.10/site-packages/'
os.environ['PATH'] = "/usr/local/bin:" + os.environ['PATH']
!python --version

## 安装依赖项 🔽

In [ ]:
%%capture
# Re-installing dependencies in the new environment context
%cd /content/ml-agents
!pip3 install -e ./ml-agents-envs
!pip3 install -e ./ml-agents

## 下载环境 zip 文件并移动到 `./trained-envs-executables/linux/` 中

- 我们的环境可执行文件位于一个 zip 文件中。
- 我们需要下载它并将其放置在 `./trained-envs-executables/linux/` 路径下。

In [ ]:
!mkdir ./trained-envs-executables
!mkdir ./trained-envs-executables/linux

我们使用 `wget` 从 https://github.com/huggingface/Huggy 下载了 Huggy.zip 文件。

In [ ]:
!wget "https://github.com/huggingface/Huggy/raw/main/Huggy.zip" -O ./trained-envs-executables/linux/Huggy.zip

In [ ]:
%%capture
!unzip -d ./trained-envs-executables/linux/ ./trained-envs-executables/linux/Huggy.zip

确保你的文件是可访问的

In [ ]:
!chmod -R 755 ./trained-envs-executables/linux/Huggy

## 让我们回顾一下这个环境是如何工作的

### 状态空间：Huggy “感知” 到了什么。

Huggy 并不是真的“看到”他的环境。相反，我们为他提供有关环境的信息：

- 目标（木棒）的位置
- 他与目标之间的相对位置
- 他腿部的方向。

有了所有这些信息，Huggy **就可以决定下一步采取什么行动来实现他的目标**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy.jpg" alt="Huggy" width="100%">


### 动作空间：Huggy 可以做出的动作
<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-action.jpg" alt="Huggy action" width="100%">

**关节电机驱动 Huggy 的腿**。这意味着为了到达目标，Huggy 需要 **学习正确旋转每条腿的关节电机，以便他可以移动**。

### 奖励函数

奖励函数的设计旨在让 **Huggy 实现他的目标**：捡回木棒。

记住，强化学习的基础之一是 *奖励假设*：目标可以描述为 **期望累积奖励的最大化**。

在这里，我们的目标是让 Huggy **走向木棒，但又不会旋转得太厉害**。因此，我们的奖励函数必须转化这个目标。

我们的奖励函数：

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/reward.jpg" alt="Huggy reward function" width="100%">

- *方向奖励*：我们 **奖励他靠近目标**。
- *时间惩罚*：每一步都给予固定的时间惩罚，以 **迫使他尽快到达木棒处**。
- *旋转惩罚*：如果 Huggy **旋转过多或转向过快**，我们会对他进行惩罚。
- *到达目标奖励*：我们奖励 Huggy **触及目标**。

## 创建 Huggy 配置文件

- 在 ML-Agents 中，你需要在 **config.yaml 文件中定义训练超参数。**

- 在本笔记本的范围内，我们不会修改超参数，但如果你想进行实验，也可以尝试修改一些其他超参数。Unity 提供了 [非常好的文档在这里解释了每一个参数](https://github.com/Unity-Technologies/ml-agents/blob/main/docs/Training-Configuration-File.md)。

- 但我们需要为 Huggy 创建一个配置文件。

  - 点击屏幕左侧的文件夹图标。

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create_file.png" alt="Create file" width="10%">

  - 前往 `/content/ml-agents/config/ppo` 目录。
  - 右键点击并创建一个名为 `Huggy.yaml` 的新文件。

  <img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/unit1/create-huggy.png" alt="Create huggy.yaml" width="20%">

- 复制并粘贴下面的内容 🔽

In [ ]:
behaviors:
  Huggy:
    trainer_type: ppo
    hyperparameters:
      batch_size: 2048
      buffer_size: 20480
      learning_rate: 0.0003
      beta: 0.005
      epsilon: 0.2
      lambd: 0.95
      num_epoch: 3
      learning_rate_schedule: linear
    network_settings:
      normalize: true
      hidden_units: 512
      num_layers: 3
      vis_encode_type: simple
    reward_signals:
      extrinsic:
        gamma: 0.995
        strength: 1.0
    checkpoint_interval: 200000
    keep_checkpoints: 15
    max_steps: 2e6
    time_horizon: 1000
    summary_freq: 50000

- 别忘了保存文件！

- **如果你想修改超参数**，在 Google Colab 笔记本中，你可以点击这里打开 config.yaml：`/content/ml-agents/config/ppo/Huggy.yaml`

- 例如，**如果你想在训练期间保存更多模型**（目前我们每 200,000 个训练步保存一次）。你需要修改：
  - `checkpoint_interval`: 每次检查点之间收集的训练步数。
  - `keep_checkpoints`: 要保留的模型检查点最大数量。

=> 请记住，**减小 `checkpoint_interval` 意味着需要上传到 Hub 的模型更多，因此上传时间也会更长。**
我们现在准备好训练我们的智能体了 🔥。

## 训练我们的智能体

要训练我们的智能体，我们只需要 **启动 mlagents-learn 并选择包含环境的可执行文件。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mllearn.png" alt="ml learn function" width="100%">

通过 ML Agents，我们运行一个训练脚本。我们定义四个参数：

1. `mlagents-learn <config>`: 超参数配置文件的路径。
2. `--env`: 环境可执行文件的位置。
3. `--run-id`: 你想给你的训练运行指定的 ID 名称。
4. `--no-graphics`: 训练期间不启动可视化窗口。

训练模型并使用 `--resume` 标志以便在中断时继续训练。

> 第一次使用 `--resume` 时可能会失败，请再次运行该代码块以跳过错误。

训练将持续 30 到 45 分钟，具体取决于你的机器（别忘了 **设置 GPU**），去喝杯咖啡吧 ☕️ 你应得的 🤗。

In [ ]:
import os
# Ensure the shell finds the installed mlagents packages
os.environ['PATH'] = "/usr/local/bin:" + os.environ['PATH']

# Start training the Huggy agent
!mlagents-learn ./config/ppo/Huggy.yaml --env=./trained-envs-executables/linux/Huggy/Huggy --run-id="Huggy_Training_Run" --no-graphics

## 将智能体推送到 🤗 Hub

- 现在我们已经训练好了智能体，我们 **准备将其推送到 Hub，以便能够在浏览器中与 Huggy 玩耍🔥。**

为了能与社区分享你的模型，还需要遵循以下三个步骤：

1️⃣ (如果尚未操作) 创建一个 HF 账号 ➡ https://huggingface.co/join

2️⃣ 登录，然后你需要存储来自 Hugging Face 网站的身份验证令牌。
- 创建一个新令牌 (https://huggingface.co/settings/tokens)，**角色选择 write**。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/create-token.jpg" alt="Create HF Token">

- 复制令牌
- 运行下面的单元格并粘贴令牌

In [ ]:
from huggingface_hub import notebook_login
notebook_login()

如果你不想使用 Google Colab 或 Jupyter Notebook，则需要使用此命令：`huggingface-cli login`

然后，我们只需要运行 `mlagents-push-to-hf`。

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/mlpush.png" alt="ml learn function" width="100%">

我们定义 4 个参数：

1. `--run-id`: 训练运行的 ID 名称。
2. `--local-dir`: 智能体保存的位置，通常是 results/<run_id 名称>。
3. `--repo-id`: 你想要创建或更新的 Hugging Face 仓库名称。格式始终为 <你的 HF 用户名>/<仓库名>。
如果仓库不存在，**它将自动创建**。
4. `--commit-message`: 既然 HF 仓库是 git 仓库，你需要定义一个提交信息。

In [ ]:
import os
# Ensure path is set
os.environ['PATH'] = "/usr/local/bin:" + os.environ['PATH']

# Push the trained agent to the Hub using your username: xwh12366
!mlagents-push-to-hf --run-id="Huggy_Training_Run" --local-dir="./results/Huggy_Training_Run" --repo-id="xwh12366/ppo-Huggy" --commit-message="Pushing my trained Huggy model"

如果一切顺利，你应该在进程结束时看到类似这样的内容（当然 URL 会有所不同 😆）：

```
Your model is pushed to the hub. You can view your model here: https://huggingface.co/ThomasSimonini/ppo-Huggy
```

这是你模型仓库的链接。该仓库包含一个解释如何使用模型的模型卡 (model card)、你的 Tensorboard 日志和你的配置文件。**最棒的是它是一个 git 仓库，这意味着你可以拥有不同的提交、通过新的推送更新你的仓库、开启 Pull Requests 等等。**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/modelcard.png" alt="ml learn function" width="100%">

但现在最精彩的部分来了：**能够在网上玩 Huggy 了 👀。**

## 与你的 Huggy 玩耍 🐕

这一步最简单：

- 在浏览器中打开 Huggy 游戏：https://huggingface.co/spaces/ThomasSimonini/Huggy

- 点击 "Play with my Huggy model" (使用我的 Huggy 模型玩耍)

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/load-huggy.jpg" alt="load-huggy" width="100%">

1. 在第 1 步中，输入你的用户名（用户名区分大小写：例如，我的用户名是 ThomasSimonini 而不是 thomassimonini 或 ThOmasImoNInI），然后点击搜索按钮。

2. 在第 2 步中，选择你的模型仓库。

3. 在第 3 步中，**选择你想回放的模型**：
  - 我有多个模型，因为我们每 500,000 步保存了一次。
  - 但既然我想要最近的，我选择 `Huggy.onnx`。

👉 尝试使用不同步数的模型来查看智能体的改进情况是很有趣的。

恭喜完成这个附加单元！

你现在可以坐下来享受与你的 Huggy 🐶 玩耍的乐趣了。别忘了通过与朋友分享 Huggy 来传递爱 🤗。如果你在社交媒体上分享，**请标记我们 @huggingface 和我 @simoninithomas**

<img src="https://huggingface.co/datasets/huggingface-deep-rl-course/course-images/resolve/main/en/notebooks/unit-bonus1/huggy-cover.jpeg" alt="Huggy cover" width="100%">


## 保持学习，保持出色 🤗